# Task 8 — Mask-based baselines con EoMT

Notebook Colab per eseguire il task 8 sugli stessi dataset anomaly del task 7, leggendo dataset e pesi da Google Drive.

Flusso:
1. Monta Drive e prepara la repo.
2. Estrae `Anomaly_Validation_Datasets.zip` in `/content`.
3. Esegue EoMT COCO / Cityscapes / fine-tuned con MSP, MaxLogit, Entropy, RbA.
4. Esegue temperature scaling.
5. Legge il CSV finale dei risultati.


## 1. Setup ambiente, Drive e repository

In [19]:
import os
import shutil
from pathlib import Path
from google.colab import drive

!pip install ood_metrics --no-deps

drive.mount("/content/drive")

# ================= CONFIGURAZIONE =================
DRIVE_ROOT = Path("/content/drive/MyDrive")

PROJECT_FOLDER = "MaskArchitectureAnomaly_CourseProject"
GIT_USERNAME = "DavideMotta"
REPO_NAME = "MaskArchitectureAnomaly_CourseProject"
BRANCH_NAME = "finetuning/coco-to-cityscapes"

project_path = DRIVE_ROOT / PROJECT_FOLDER
repo_url = f"https://github.com/{GIT_USERNAME}/{REPO_NAME}.git"
# ==================================================


def setup_repository():
    print(f"\n--- Gestione progetto: {PROJECT_FOLDER} ---")

    if not project_path.exists():
        print("📂 Clonazione repository...")
        os.system(f'git clone --branch "{BRANCH_NAME}" "{repo_url}" "{project_path}"')
    else:
        git_dir = project_path / ".git"

        if not git_dir.exists():
            print("⚠️ Cartella esistente ma non è una repo Git. La ricreo...")
            shutil.rmtree(project_path)
            os.system(f'git clone --branch "{BRANCH_NAME}" "{repo_url}" "{project_path}"')
        else:
            print("🔄 Aggiornamento repository...")
            os.chdir(project_path)
            os.system("git fetch origin")
            os.system(f"git checkout {BRANCH_NAME}")
            os.system(f"git reset --hard origin/{BRANCH_NAME}")

    print("✅ Repository pronta.")
    print("Percorso progetto:", project_path)


setup_repository()
os.chdir(project_path)

# Dipendenze EoMT
!pip install -r eomt/requirements.txt


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Gestione progetto: MaskArchitectureAnomaly_CourseProject ---
🔄 Aggiornamento repository...
✅ Repository pronta.
Percorso progetto: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject


## 2. Path principali: repo, EoMT, dataset, pesi

In [20]:
from pathlib import Path
import zipfile
import glob
import os

PROJECT_ROOT = Path.cwd()
MYDRIVE_ROOT = PROJECT_ROOT.parents[0]

# Nella repo del corso, EoMT è nella sottocartella eomt/
EOMT_ROOT = PROJECT_ROOT / "eomt"

# Eval folder storica del task 7
EVAL_DIR = PROJECT_ROOT / "eval"
EVAL_DIR.mkdir(exist_ok=True)

# Zip dei dataset anomaly, come nel notebook Task 7
ANOMALY_ZIP = (
    MYDRIVE_ROOT
    / "FAIML_project_and_presentation"
    / "01_Project"
    / "large_files"
    / "datasets"
    / "anomaly"
    / "Anomaly_Validation_Datasets.zip"
)

LOCAL_DATA_DIR = Path("/content/anomaly_data")
DATA_ROOT = LOCAL_DATA_DIR / "Validation_Dataset"

# Cartella pesi
WEIGHTS_ROOT = (
    MYDRIVE_ROOT
    / "FAIML_project_and_presentation"
    / "01_Project"
    / "large_files"
    / "weights"
)

# Output CSV su Drive, così non lo perdi se il runtime si resetta
RESULTS_CSV = (
    MYDRIVE_ROOT
    / "FAIML_project_and_presentation"
    / "01_Project"
    / "results_task8_eomt_V4_with_stage0_model_class_n_mask_heads.csv"
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EOMT_ROOT:", EOMT_ROOT, "| exists:", EOMT_ROOT.exists())
print("EVAL_DIR:", EVAL_DIR, "| exists:", EVAL_DIR.exists())
print("ANOMALY_ZIP:", ANOMALY_ZIP, "| exists:", ANOMALY_ZIP.exists())
print("WEIGHTS_ROOT:", WEIGHTS_ROOT, "| exists:", WEIGHTS_ROOT.exists())
print("RESULTS_CSV:", RESULTS_CSV)


PROJECT_ROOT: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject
EOMT_ROOT: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt | exists: True
EVAL_DIR: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eval | exists: True
ANOMALY_ZIP: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/datasets/anomaly/Anomaly_Validation_Datasets.zip | exists: True
WEIGHTS_ROOT: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights | exists: True
RESULTS_CSV: /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/results_task8_eomt_V4_with_stage0_model_class_n_mask_heads.csv


## 3. Estrazione dataset anomaly in `/content`

Si estrae in `/content` per evitare di leggere continuamente immagini dallo zip/Drive durante l'inference.


In [21]:
print("Zip exists:", ANOMALY_ZIP.exists())

if not LOCAL_DATA_DIR.exists():
    print("Extracting zip temporarily to /content...")
    with zipfile.ZipFile(ANOMALY_ZIP, "r") as z:
        z.extractall(LOCAL_DATA_DIR)
    print("Done.")
else:
    print("Already extracted in this runtime.")

print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("Contenuto DATA_ROOT:")
for p in sorted(DATA_ROOT.iterdir()) if DATA_ROOT.exists() else []:
    print("-", p.name)


Zip exists: True
Already extracted in this runtime.
DATA_ROOT: /content/anomaly_data/Validation_Dataset
DATA_ROOT exists: True
Contenuto DATA_ROOT:
- .DS_Store
- FS_LostFound_full
- RoadAnomaly
- RoadAnomaly21
- RoadObsticle21
- fs_static


## 4. Definizione dataset anomaly e controllo numero immagini

In [22]:
datasets = {
    "FS_LostFound_full": DATA_ROOT / "FS_LostFound_full" / "images" / "*.png",
    "fs_static": DATA_ROOT / "fs_static" / "images" / "*.jpg",
    "RoadAnomaly": DATA_ROOT / "RoadAnomaly" / "images" / "*.jpg",
    "RoadAnomaly21": DATA_ROOT / "RoadAnomaly21" / "images" / "*.png",
    "RoadObsticle21": DATA_ROOT / "RoadObsticle21" / "images" / "*.webp",
}

for name, pattern in datasets.items():
    files = glob.glob(str(pattern))
    print(name, len(files), "images")


FS_LostFound_full 100 images
fs_static 30 images
RoadAnomaly 60 images
RoadAnomaly21 10 images
RoadObsticle21 30 images


## 5. Checkpoint EoMT

Aggiorna `EOMT_FINETUNED_WEIGHTS` se il tuo checkpoint fine-tuned ha un nome/path diverso.

I preset da usare sono:
- `coco` → `num_classes=133`, `num_q=200`
- `cityscapes` → `num_classes=19`, `num_q=100`
- `finetuned` → `num_classes=19`, `num_q=100`


In [23]:
EOMT_COCO_WEIGHTS = WEIGHTS_ROOT / "eomt_coco.bin"
EOMT_CITYSCAPES_WEIGHTS = WEIGHTS_ROOT / "eomt_cityscapes.bin"
checkpoints = {
    "eomt_coco": {
        "weights": EOMT_COCO_WEIGHTS,
        "preset": "coco",
    },
     "eomt_cityscapes": {
         "weights": EOMT_CITYSCAPES_WEIGHTS,
         "preset": "cityscapes",
     },
     "eomt_finetuned_stage0": {
         "weights": WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_stage0_heads" / "best.ckpt",
         "preset": "finetuned",
     },
     "eomt_finetuned_stage1": {
         "weights": WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_stage1_head" / "stage1_weights.bin",
         "preset": "finetuned",
     },
     "eomt_finetuned_stage2": {
         "weights": WEIGHTS_ROOT / "finetuned" / "coco_to_cityscapes_stage2_unfreeze_last" / "stage2_weights.bin",
         "preset": "finetuned",
     },
}

for name, cfg in checkpoints.items():
    print(name, "->", cfg["weights"], "| exists:", cfg["weights"].exists())

eomt_coco -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/eomt_coco.bin | exists: True
eomt_cityscapes -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/eomt_cityscapes.bin | exists: True
eomt_finetuned_stage0 -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_stage0_heads/best.ckpt | exists: True
eomt_finetuned_stage1 -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_stage1_head/stage1_weights.bin | exists: True
eomt_finetuned_stage2 -> /content/drive/MyDrive/FAIML_project_and_presentation/01_Project/large_files/weights/finetuned/coco_to_cityscapes_stage2_unfreeze_last/stage2_weights.bin | exists: True


In [ ]:
# Verifica num_q / num_classes leggendoli direttamente dai checkpoint.
# num_q  = q.weight.shape[0]   (numero di mask queries imparate dal modello)
# num_classes = class_head.weight.shape[0] - 1   (-1 per la classe "no-object")
import torch

def inspect_ckpt(path):
    sd = torch.load(path, map_location="cpu")
    sd = sd["state_dict"] if isinstance(sd, dict) and "state_dict" in sd else sd
    num_q = num_classes = None
    for k, v in sd.items():
        kk = k.replace("._orig_mod", "").replace("network.", "").replace("module.", "")
        if kk.endswith("q.weight"):
            num_q = v.shape[0]
        if kk.endswith("class_head.weight"):
            num_classes = v.shape[0] - 1
    return num_q, num_classes

PRESET_EXPECTED = {"coco": (200, 133), "cityscapes": (100, 19), "finetuned": (200, 19)}

for name, cfg in checkpoints.items():
    if not cfg["weights"].exists():
        print(f"{name}: pesi non trovati"); continue
    q, c = inspect_ckpt(cfg["weights"])
    exp_q, exp_c = PRESET_EXPECTED[cfg["preset"]]
    ok = (q == exp_q) and (c == exp_c)
    flag = "OK" if ok else "!!! MISMATCH col preset"
    print(f"{name:28s} preset={cfg['preset']:11s} num_q={q} num_classes={c} "
          f"(atteso {exp_q}/{exp_c}) {flag}")


### 6.1 Pre-caricamento dei pesi dei checkpoint in memoria

Per evitare di ricaricare i pesi da disco più volte, carichiamo il `state_dict` di ogni checkpoint in memoria. Tuttavia, lo script `evalAnomaly_eomt.py` (eseguito tramite `subprocess`) caricherà comunque i pesi dal path fornito, poiché opera in un processo separato.

Per sfruttare questi pesi pre-caricati, lo script `evalAnomaly_eomt.py` dovrebbe essere modificato per accettare un `state_dict` già in memoria o per implementare una cache interna, oppure la logica di valutazione dovrebbe essere integrata direttamente in questo notebook.

In [ ]:
# import torch
# import sys

# total_weights_size_mb = 0
# print("--- Pre-caricamento pesi in memoria ---")

# for name, cfg in checkpoints.items():
#     if cfg["weights"].exists():
#         print(f"Caricamento pesi per {name}...")
#         # Carica il state_dict su CPU per non occupare memoria GPU finché non serve
#         loaded_state_dict = torch.load(cfg["weights"], map_location='cpu')
#         cfg["loaded_state_dict"] = loaded_state_dict

#         # Calcola la dimensione del state_dict in memoria
#         # Approssimazione: dimensione di ogni tensore * numero di elementi * dimensione in byte del tipo
#         # Questo è un valore stimato, il consumo reale potrebbe variare.
#         size_bytes = 0
#         for k, v in loaded_state_dict.items():
#             size_bytes += v.element_size() * v.nelement()
#         size_mb = size_bytes / (1024 * 1024)
#         total_weights_size_mb += size_mb
#         print(f"  -> Caricato. Dimensione approssimativa: {size_mb:.2f} MB")
#     else:
#         print(f"Skip {name}: pesi non trovati a {cfg['weights']}")

# print(f"--- Totale pesi pre-caricati in memoria: {total_weights_size_mb:.2f} MB ---")


## 6. Funzione per lanciare una singola evaluation

Uso `subprocess.run` con `PYTHONPATH` impostato in modo che lo script in `eval/` trovi i moduli EoMT dentro `eomt/models`.


In [24]:
# Path dello script di evaluation EoMT
SCRIPT_PATH = PROJECT_ROOT / "eval" / "evalAnomaly_eomt.py"

assert SCRIPT_PATH.exists(), f"Script non trovato: {SCRIPT_PATH}"

print("Script trovato:", SCRIPT_PATH)

Script trovato: /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eval/evalAnomaly_eomt.py


In [ ]:
# =====================================================================
# Motore di valutazione IN-PROCESS (niente subprocess).
#
# Idea: per ogni checkpoint il modello si carica UNA volta, e per ogni
# dataset il forward si esegue UNA volta per immagine. I 4 metodi
# (msp/maxlogit/entropy/rba) e le varie temperature sono solo
# post-processing degli stessi logit -> niente forward ripetuti.
#
# Riusa le funzioni gia' testate di eval/evalAnomaly_eomt.py e la stessa
# save_csv -> il CSV in output e' identico a prima.
# =====================================================================
import sys, glob, os
from types import SimpleNamespace
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr

for p in [str(EOMT_ROOT), str(PROJECT_ROOT), str(EVAL_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

import evalAnomaly_eomt as E  # build_eomt, load_eomt_weights, score fns, save_csv, transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = True  # autocast fp16 sul forward: ~2x piu' veloce. Metti False per match esatto coi run fp32.
print("Device:", DEVICE, "| AMP fp16:", USE_AMP)


def build_model(weights_path, preset):
    args = SimpleNamespace(
        preset=preset, num_blocks=3, patch_size=16,
        backbone_name="vit_base_patch14_reg4_dinov2",
    )
    args = E.apply_preset(args)
    model = E.build_eomt(args)
    model = E.load_eomt_weights(model, str(weights_path))
    return model.to(DEVICE).eval(), args


@torch.no_grad()
def eval_checkpoint(checkpoint_name, weights_path, preset, methods, temperatures=(1.0,)):
    """Carica il modello una volta e valuta tutti i dataset/metodi/temperature."""
    model, args = build_model(weights_path, preset)
    target_size = (E.IMG_HEIGHT, E.IMG_WIDTH)

    for dataset_name, pattern in datasets.items():
        paths = sorted(glob.glob(str(pattern)))
        if not paths:
            print(f"SKIP dataset vuoto: {dataset_name}")
            continue

        # rba ignora la temperatura -> sempre solo T=1.0
        runs = [(m, T) for m in methods for T in (temperatures if m != "rba" else (1.0,))]
        scores = {k: [] for k in runs}
        gts = {k: [] for k in runs}

        print(f"\n[{checkpoint_name}] {dataset_name}: {len(paths)} immagini")
        for path in paths:
            img = E.input_transform(Image.open(path).convert("RGB")).unsqueeze(0).float().to(DEVICE)
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=(USE_AMP and DEVICE.type == "cuda")):
                ml_layers, cl_layers = model(img)
            ml, cl = ml_layers[-1].float(), cl_layers[-1].float()  # forward calcolato UNA volta

            pathGT = E.get_gt_path(path)
            if not os.path.exists(pathGT):
                continue
            gt = E.convert_gt(np.array(E.target_transform(Image.open(pathGT))), pathGT)
            if 1 not in np.unique(gt):
                continue
            valid = (gt == 0) | (gt == 1)
            gt_v = gt[valid].astype(np.uint8)

            pix = E.eomt_to_pixel_logits(ml, cl, target_size)  # condiviso da msp/maxlogit/entropy
            for (m, T) in runs:
                if m == "rba":
                    s = E.compute_rba_score(ml, cl, target_size)
                else:
                    s = E.compute_anomaly_score(pix, m, T)
                s = s.squeeze(0).float().cpu().numpy()[valid].astype(np.float32)
                scores[(m, T)].append(s)
                gts[(m, T)].append(gt_v)

        for (m, T) in runs:
            if not gts[(m, T)]:
                continue
            label = np.concatenate(gts[(m, T)])
            out = np.concatenate(scores[(m, T)])
            auprc = average_precision_score(label, out)
            fpr95 = fpr_at_95_tpr(out, label)

            # riempie args e riusa la save_csv originale -> stesse colonne del CSV
            args.checkpoint_name = checkpoint_name
            args.weights = str(weights_path)
            args.input = [str(pattern)]
            args.method = m
            args.temperature = T
            args.results_csv = str(RESULTS_CSV)
            E.save_csv(
                args, auprc, fpr95,
                num_images=len(gts[(m, T)]),
                num_pixels=len(label),
                num_anomaly_pixels=int(label.sum()),
            )
            print(f"  {m:8s} T={T:<4} -> AUPRC={auprc*100:.2f}  FPR95={fpr95*100:.2f}")

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()


## 7. Test su un solo dataset/metodo

Prima di lanciare tutto, fai un test piccolo. Consiglio: `RoadAnomaly` + `eomt_cityscapes` + `msp`.


In [26]:
RUN_SMOKE_TEST = False # Impostato a False per saltare il test rapido

## 8. Run completo Task 8: 3 checkpoint × 5 dataset × 4 metodi

Lascia `RUN_FULL = False` finché lo smoke test non funziona.  
Poi metti `RUN_FULL = True`.

Metodi:
- `msp`
- `maxlogit`
- `entropy`
- `rba`


In [ ]:
RUN_FULL = True

methods = ["msp", "maxlogit", "entropy", "rba"]

if RUN_FULL:
    for checkpoint_name, cfg in checkpoints.items():
        if not cfg["weights"].exists():
            print(f"Skip {checkpoint_name}: weights non trovati")
            continue
        # 1 load del modello + 1 forward per immagine; tutti i metodi dai logit in cache
        eval_checkpoint(
            checkpoint_name=checkpoint_name,
            weights_path=cfg["weights"],
            preset=cfg["preset"],
            methods=methods,
        )
else:
    print("RUN_FULL è False. Cambialo a True per lanciare il run completo.")


##9. Temperature scaling

Il task chiede di provare temperature scaling.  
Qui lo applichiamo a `MSP`.

Puoi scegliere checkpoint e dataset. Per il run completo, puoi ciclare su tutti i dataset.


In [ ]:
RUN_TEMPERATURE = False

temperatures = [0.5, 0.75, 1.0, 1.1]

TEMP_CHECKPOINT_NAME = "eomt_cityscapes"

if RUN_TEMPERATURE:
    cfg = checkpoints[TEMP_CHECKPOINT_NAME]
    # Un solo forward per immagine: le diverse T sono solo post-processing sui logit.
    # La temperatura finisce nella colonna "temperature" del CSV (la pivot la usa gia').
    eval_checkpoint(
        checkpoint_name=TEMP_CHECKPOINT_NAME,
        weights_path=cfg["weights"],
        preset=cfg["preset"],
        methods=["msp"],
        temperatures=temperatures,
    )
else:
    print("RUN_TEMPERATURE è False. Cambialo a True per lanciare temperature scaling.")


## 10. Lettura risultati CSV

In [ ]:
import pandas as pd

if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)
    display(df.tail(30))
else:
    print("CSV non ancora creato:", RESULTS_CSV)


## 11. Tabella pivot per report

Questa cella crea una tabella leggibile con righe `checkpoint/method/temperature` e colonne per dataset.


In [ ]:
if RESULTS_CSV.exists():
    df = pd.read_csv(RESULTS_CSV)

    # Ricava nome dataset dal path input
    def infer_dataset_name(input_path):
        p = str(input_path)
        for name in datasets.keys():
            if name in p:
                return name
        return "unknown"

    df["dataset"] = df["input"].apply(infer_dataset_name)

    pivot_auprc = df.pivot_table(
        index=["checkpoint_name", "preset", "method", "temperature"],
        columns="dataset",
        values="AUPRC",
        aggfunc="last",
    )

    pivot_fpr95 = df.pivot_table(
        index=["checkpoint_name", "preset", "method", "temperature"],
        columns="dataset",
        values="FPR95",
        aggfunc="last",
    )

    print("AUPRC")
    display(pivot_auprc)

    print("FPR95")
    display(pivot_fpr95)
else:
    print("CSV non ancora creato.")


In [ ]:
if RESULTS_CSV.exists():
    # Define the output Excel file path
    EXCEL_OUTPUT_PATH = RESULTS_CSV.parent / "task8_results.xlsx"

    # Create a Pandas Excel writer using XlsxWriter as the engine.
    # The 'with' statement ensures the writer is closed automatically.
    with pd.ExcelWriter(EXCEL_OUTPUT_PATH, engine='xlsxwriter') as writer:
        # Write each DataFrame to a different worksheet.
        pivot_auprc.to_excel(writer, sheet_name='AUPRC')
        pivot_fpr95.to_excel(writer, sheet_name='FPR95')

    print(f"Risultati AUPRC e FPR95 salvati in Excel: {EXCEL_OUTPUT_PATH}")
else:
    print("CSV non ancora creato. Impossibile salvare in Excel.")